In [ ]:
### Cell 1 — Load All Results
# ⚠️ rf_results.json is produced by Ahmed (notebooks/03_random_forest.ipynb).
# Run that notebook first, then re-run this cell.

import json, os
import matplotlib.pyplot as plt
import numpy as np

REPORT_DIR = "../report"

def load_json(path, label):
    if not os.path.exists(path):
        print(f"⚠️  {label} not found at {path} — skipping (run the corresponding notebook first).")
        return None
    with open(path) as f:
        return json.load(f)

lr = load_json(f"{REPORT_DIR}/lr_results.json", "lr_results.json")
rf = load_json(f"{REPORT_DIR}/rf_results.json", "rf_results.json")
gb = load_json(f"{REPORT_DIR}/gb_results.json", "gb_results.json")

available = {name: res for name, res in
             [("Linear Regression", lr), ("Random Forest", rf), ("Gradient Boosting", gb)]
             if res is not None}

if not available:
    raise RuntimeError("No result files found. Run notebooks 02, 03, 04 first.")

models = list(available.keys())
maes   = [available[m]["mae"]  for m in models]
rmses  = [available[m]["rmse"] for m in models]

print(f"{chr(10)}Results loaded: {list(available.keys())}")
print(f"{"Model":<22} {"MAE":>8} {"RMSE":>8}")
print("-" * 42)
for m, mae, rmse in zip(models, maes, rmses):
    print(f"{m:<22} {mae:>8.4f} {rmse:>8.4f}")

best = models[maes.index(min(maes))]
print(f"
✅ Best model by MAE: {best}")

if len(available) < 3:
    missing = [m for m in ["Linear Regression", "Random Forest", "Gradient Boosting"] if m not in available]
    print(f"
⚠️  Still waiting on: {missing}")
    print("    Comparison chart will be partial until all 3 results are available.")


In [ ]:
### Cell 2 — MAE/RMSE Comparison Bar Chart
# Works with 1, 2 or 3 models — partial chart until all results are in.

all_colors_mae  = ["#4c72b0", "#55a868", "#c44e52"]
all_colors_rmse = ["#8da0cb", "#a6d96a", "#f4a582"]
n = len(models)
colors_mae  = all_colors_mae[:n]
colors_rmse = all_colors_rmse[:n]

x = np.arange(n)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, maes,  width, label="MAE",  color=colors_mae,  edgecolor="black")
bars2 = ax.bar(x + width/2, rmses, width, label="RMSE", color=colors_rmse, edgecolor="black")

ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Error", fontsize=12)
title = "Model Comparison — MAE & RMSE on Test Set"
if n < 3:
    title += f" ({n}/3 models)"
ax.set_title(title, fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.legend(fontsize=11)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)

os.makedirs("../report/figures", exist_ok=True)
plt.tight_layout()
plt.savefig("../report/figures/model_comparison.png", dpi=150)
plt.show()
print("Saved → report/figures/model_comparison.png")


In [ ]:
### Cell 3 — Summary Table for Report

print("
=== TABLE FOR REPORT ===")
print(f"{chr(10)}{"Model":<22} | {"MAE":>8} | {"RMSE":>8} | vs Baseline")
print("-" * 60)

base_mae = lr["mae"] if lr else None

for m, mae, rmse in zip(models, maes, rmses):
    if base_mae is not None:
        delta = (base_mae - mae) / base_mae * 100
        direction = f"▼ {delta:.1f}%" if delta > 0 else f"▲ {abs(delta):.1f}%"
    else:
        direction = "(no LR baseline)"
    print(f"{m:<22} | {mae:>8.4f} | {rmse:>8.4f} | {direction}")

if len(models) < 3:
    print(f"
⚠️  Table is incomplete — run missing notebooks to add remaining models.")
